# Exercises: Heuristic Methods and Stochastic Games

**Topic:** Adversarial Search and Games  
**Focus:** Heuristic Methods, Monte Carlo Tree Search, and Stochastic Games

---

- **Id:** 22302421
- **Full name:** Nguyen Phuc Minh Chau

---
## Learning Objectives

After completing this notebook, students should be able to:

1. Explain why exact minimax search becomes impractical in large game trees.
2. Describe heuristic cutoff search and forward pruning.
3. Design a simple evaluation function for a game state.
4. Explain the intuition of Monte Carlo Tree Search (MCTS).
5. Compute UCB1 values and discuss exploration vs. exploitation.
6. Explain stochastic games and chance nodes.
7. Solve simple expectiminimax problems.
8. Compare heuristic alpha-beta search and MCTS in stochastic settings.


## Part 1. Short Review Questions

Answer the following in your own words.

### Question 1
Why are heuristic methods needed in adversarial games?

**Your answer:**

Heuristic methods are needed in adversarial games because these games often have a large search space, making it computationally infeasible to explore all possible moves and outcomes. Heuristics provide a way to evaluate the game state and make informed decisions without having to exhaustively search through every possible move. They help players prioritize certain moves over others based on factors such as potential threats, strategic advantages, and the overall game state, allowing for more efficient and effective gameplay.

---

### Question 2
What is the main idea of **cutoff search** in heuristic alpha-beta tree search?

**Your answer:**

Cutoff search is a technique used in heuristic alpha-beta tree search to limit the depth of the search tree. The main idea is to set a cutoff depth, beyond which the search will not continue. When the search reaches this cutoff depth, it evaluates the position using a heuristic evaluation function instead of continuing to explore further down the tree. This allows the algorithm to focus on more promising branches of the search tree while avoiding the computational cost of exploring less promising branches in depth.

---

### Question 3
What is **forward pruning**? What is its main risk?

**Your answer:**

Forward pruning is a technique used in search algorithms, particularly in game tree search, where certain branches of the search tree are pruned or ignored based on heuristics or evaluation functions. The main risk of forward pruning is that it can lead to missing critical moves or strategies, as it may prune branches that could have led to a winning position. This can result in suboptimal decisions and potentially losing the game.

---

### Question 4
Why is a good evaluation function important in heuristic search?

**Your answer:**

A good evaluation function is crucial in heuristic search because it guides the search process towards the most promising paths, helping to efficiently find a solution. It estimates the cost or distance to the goal, allowing the algorithm to prioritize which nodes to explore next. A well-designed evaluation function can significantly reduce the search space and improve the performance of the algorithm, while a poor evaluation function may lead to inefficient searches and longer solution times.

## Part 2. Heuristic Evaluation Function

In cutoff search, we stop before reaching terminal states and estimate state quality using an evaluation function:

$$ Eval(s) = w_1 f_1(s) + w_2 f_2(s) + \dots + w_n f_n(s) $$

Suppose we are evaluating a simplified board game state using:

- `f1(s)` = number of Max pieces advantage
- `f2(s)` = number of possible winning lines for Max
- `f3(s)` = number of possible winning lines for Min

and the evaluation function is:

$$Eval(s) = 2f_1(s) + 3f_2(s) - 4f_3(s)$$

### Exercise 2.1
Compute `Eval(s)` for the following state:

- `f1(s) = 2`
- `f2(s) = 3`
- `f3(s) = 1`

**Show your calculation here:**

$$Eval(s) = 2*2 + 3*3 - 4*1 = 9$$
---

### Exercise 2.2
Compute `Eval(s)` for this state:

- `f1(s) = -1`
- `f2(s) = 4`
- `f3(s) = 2`

**Show your calculation here:**

$$Eval(s) = 2*-1 + 3*4 - 4*2 = 2$$
---

### Exercise 2.3
Explain the meaning of a **large positive** evaluation score and a **large negative** evaluation score.

**Your answer:**

A large positive evaluation score indicates that the position is favorable for the player, while a large negative evaluation score indicates that the position is unfavorable for the player.

In other words, a large positive score suggests that the player has a strong advantage, while a large negative score suggests that the player is at a disadvantage

---
### Exercise 2.4
Write a Python function to compute the evaluation score.

In [1]:
def eval_state(f1, f2, f3):
    # TODO: return 2*f1 + 3*f2 - 4*f3
    return 2*f1 + 3*f2 - 4*f3
    pass

# Test your function with the two examples above.
print(eval_state(2, 3, 1))
print(eval_state(-1, 4, 2))

9
2


## Part 3. Cutoff Search Reasoning

In heuristic alpha-beta tree search, we may stop search at a fixed depth and use `Eval(s)` instead of the true utility.

Consider this simplified depth-2 game tree for Max:

- Action `a1` leads to a Min node with leaf evaluations: `3, 5, 2`
- Action `a2` leads to a Min node with leaf evaluations: `4, 1, 6`
- Action `a3` leads to a Min node with leaf evaluations: `0, 7, 8`

### Exercise 3.1
Compute the heuristic minimax value (HMV) for each action.

Remember:
- Max chooses the maximum value.
- Min chooses the minimum value.

**Your work:**

- `HMV(a1) =`min(a1) = min(3, 5, 2) = 2
- `HMV(a2) =`min(a2) = min(4, 1, 6) = 1
- `HMV(a3) =`min(a3) = min(0, 7, 8) = 0

---

### Exercise 3.2
Which action should Max choose?

**Your answer:**

HMV(a1) = 2
---

### Exercise 3.3
Why can cutoff search produce a different decision from full minimax search?

**Your answer:**

Cutoff search stops at a fixed depth and replaces true utility with Eval(s) — a heuristic estimate. This can produce a different decision for two reasons:

1. The evaluation function is an approximation.

    Eval(s) estimates how good a state is, but it is not the true outcome of the game. A state that looks strong at depth 2 may actually lead to a losing position several moves deeper. The heuristic cannot see past the cutoff horizon.

2. The horizon effect.

    A dangerous or decisive event (a capture, a blunder, a forced sequence) may lie just beyond the cutoff depth. Because the search cannot see it, Eval(s) assigns a misleading value to that state, potentially causing Max to choose the wrong action.

In short: cutoff search trades completeness and optimality for computational feasibility. The quality of the decision is only as good as the evaluation function's ability to approximate the true game outcome at the cutoff states.

## Part 4. Forward Pruning

Forward pruning keeps only a small number of promising moves and ignores the rest.

Suppose a shallow heuristic search gives these preliminary scores for Max's available moves:

- `a1 = 4.2`
- `a2 = 1.8`
- `a3 = 5.0`
- `a4 = -0.5`
- `a5 = 3.7`

### Exercise 4.1
If beam width = 2, which moves are kept?

**Your answer:**
 
a3, a1

---

### Exercise 4.2
If beam width = 3, which moves are kept?

**Your answer:**

a3, a1, a5

---

### Exercise 4.3
What important mistake can forward pruning make?

**Your answer:**

Forward pruning can permanently discard the true optimal move. A move that scores poorly under the shallow heuristic may actually lead to the best outcome at deeper levels — the shallow evaluation simply cannot see far enough to recognize its potential. Once a move is pruned, it is never reconsidered, so the error is unrecoverable. This is called the pruning of the best move.

In this example if beam width = 2, moves a5, a2, and a4 are discarded entirely. If the true minimax optimal move happened to be a5 (score 3.7), it would be lost with no warning.

---

### Exercise 4.4
Give one advantage and one disadvantage of forward pruning.

**Advantage:**  

Drastically reduces the branching factor, allowing the search to go deeper within the same time budget. Fewer moves to explore at each node means computational resources are concentrated on the most promising lines.

**Disadvantage:**

Sacrifices optimality guarantees. Unlike alpha-beta pruning (which is provably safe), forward pruning can eliminate the best move based on a misleading shallow score, leading the agent to a suboptimal or even losing decision with no way to detect the error.

---

### Exercise 4.5
Complete the function below to return the top-k moves by score.

In [ ]:
def top_k_moves(move_scores, k):
    """
    move_scores: dictionary, e.g. {'a1': 4.2, 'a2': 1.8}
    k: number of moves to keep
    """
    # TODO
    move_scores_sorted = sorted(move_scores.items(),
                                 key=lambda x: x[1], reverse=True)
    top_k = move_scores_sorted[:k]
    return [move for move, score in top_k]
    pass

scores = {'a1': 4.2, 'a2': 1.8, 'a3': 5.0, 'a4': -0.5, 'a5': 3.7}
print(top_k_moves(scores, 2))
print(top_k_moves(scores, 3))

['a3', 'a1']
['a3', 'a1', 'a5']


## Part 5. Monte Carlo Search and MCTS

Monte Carlo methods estimate the value of a move by simulating many playouts and averaging the outcomes.

Suppose the following moves have these win statistics after random playouts:

| Move | Wins | Playouts |
|------|------|----------|
| a1   | 27   | 50       |
| a2   | 18   | 30       |
| a3   | 31   | 40       |
| a4   | 12   | 20       |

### Exercise 5.1
Compute the empirical win rate for each move.

**Your work:**

For winrate = win/playouts

- `a1 = 27/50 = 0.54`
- `a2 =`18/30 = 0.6
- `a3 =`31/40 = 0.775
- `a4 =`12/20 = 0.6

---

### Exercise 5.2
Which move looks best using pure Monte Carlo search?

**Your answer:**

It's a3 with the highest result: 0.775

---

### Exercise 5.3
Why might pure Monte Carlo search waste time?

**Your answer:**

Pure Monte Carlo distributes playouts uniformly or randomly across all moves, regardless of how promising each one looks. This means it spends just as many simulations on clearly weak moves as on strong ones. Computational budget is wasted on moves that a smarter policy would have deprioritized early — the search has no mechanism to concentrate effort where it matters most.

---

### Exercise 5.4
What is the difference between **pure Monte Carlo search** and **Monte Carlo Tree Search (MCTS)**?

**Your answer:**

Pure Monte Carlo search evaluates moves by simulating random playouts from the current state and averaging the results, without building a search tree. Monte Carlo Tree Search (MCTS), on the other hand, builds a search tree where each node represents a game state, and edges represent possible moves. MCTS uses a selection strategy to explore the tree, balancing exploration of new moves and exploitation of known good moves, which allows it to focus on promising areas of the search space.

---

### Exercise 5.5
Compute win rates from the given data.


In [3]:
moves = {
    'a1': {'wins': 27, 'playouts': 50},
    'a2': {'wins': 18, 'playouts': 30},
    'a3': {'wins': 31, 'playouts': 40},
    'a4': {'wins': 12, 'playouts': 20}
}

for move, stats in moves.items():
    # TODO: compute and print win rate
    #monte carlo tree search
    win_rate = stats['wins'] / stats['playouts']
    print(f"{move}: {win_rate:.2f}")
    pass

a1: 0.54
a2: 0.60
a3: 0.78
a4: 0.60


## Part 6. UCB1 and the Exploration-Exploitation Tradeoff

In MCTS, UCB1 helps choose which node to explore next:

$$UCB1(n) = \frac{U(n)}{N(n)} + C \sqrt{\frac{\log N(parent(n))}{N(n)}}$$

where:

- `U(n)` = total utility of playouts through node `n`
- `N(n)` = number of playouts through node `n`
- `N(parent(n))` = number of playouts through the parent node
- `C` = exploration constant

Assume:

- `C = 2`
- `N(parent) = 100`

and two child nodes:

- Node A: `U(A)=30`, `N(A)=40`
- Node B: `U(B)=12`, `N(B)=10`

### Exercise 6.1
Compute the average utility of A and B.

**Your work:**

- `Avg(A) =`U(A)/N(A) = 30/40 = 0.75
- `Avg(B) =`U(B)/N(B) = 12/10 = 1.2

---

### Exercise 6.2
Without fully calculating the square root term, which node is likely to get a larger exploration bonus? Why?

**Your answer:**

The exploration term is:

 $$C \sqrt{\frac{\log N(parent(n))}{N(n)}}​$$

logN(parent)=log100 is the same for both nodes. The only difference is N(n) in the denominator:


Node A: N(A)=40 → smaller fraction → smaller bonus
Node B:N(B)=10 → larger fraction → larger bonus

Node B gets the larger exploration bonus because it has been visited far fewer times. UCB1 is designed this way intentionally — under-explored nodes receive a higher bonus to encourage the search to investigate them before drawing conclusions.

---

### Exercise 6.3
Explain in words what **exploration** and **exploitation** mean in MCTS.

**Your answer:**

Exploration refers to the strategy of trying out new or less frequently visited moves in the search tree to discover potentially better outcomes. It encourages the algorithm to gather more information about the game state and avoid getting stuck in local optima. 

Exploitation, on the other hand, focuses on selecting moves that have already shown promising results based on previous simulations. It leverages the accumulated knowledge to maximize the chances of winning. 

The UCB1 formula balances these two aspects by assigning a score to each move that considers both its average reward (exploitation) and the uncertainty or potential for improvement (exploration)

---

### Exercise 6.4
Implement UCB1.

In [6]:
import math

def ucb1(U, N, parent_N, C=2.0):
    # TODO: implement the formula
    #UCB1 = U/N + C * sqrt(log(parent_N)/N)
    if N == 0:
        return float('inf')  # If a move has never been tried, return infinity to ensure it gets selected
    return U/N + C * math.sqrt(math.log(parent_N)/N)
    pass

print('UCB1(A)=', ucb1(30, 40, 100, 2.0))
print('UCB1(B)=', ucb1(12, 10, 100, 2.0))

UCB1(A)= 1.4286140424415112
UCB1(B)= 2.5572280848830227


## Part 7. Stochastic Games

A stochastic game includes random events such as dice rolls or card draws. These are represented by **chance nodes**.

### Question 7.1
Give two examples of random events in games.

**Your answer:**

1. In Monopoly, rolling the dice to determine how many spaces a player moves is a random event.

2. In Poker, the dealing of cards to players is a random event.

---

### Question 7.2
Why can stochastic games be harder than deterministic games?

**Your answer:**

Stochastic games can be harder than deterministic games because they involve uncertainty and randomness, which can make it more difficult to predict outcomes and plan strategies. In deterministic games, the outcome of each action is known and predictable, allowing players to make informed decisions based on the current state of the game. 

In contrast, stochastic games require players to consider probabilities and potential outcomes of random events, which can lead to a larger search space and increased complexity in decision-making. Additionally, players may need to account for risk and uncertainty when evaluating different strategies, making it more challenging to find optimal solutions.

---

### Question 7.3
What does a chance node compute in expectiminimax?

**Your answer:**

A chance node computes the expected value of its child nodes, weighted by the probabilities of the random events.

## Part 8. Expectiminimax Practice

Consider a chance node with three possible outcomes:

- Outcome `r1` with probability `0.2` leads to value `8`
- Outcome `r2` with probability `0.5` leads to value `2`
- Outcome `r3` with probability `0.3` leads to value `-4`

### Exercise 8.1
Compute the expected value of this chance node.

$$Expected\ Value = \sum_r P(r) \cdot Value(r)$$

**Your calculation:**
$$EV=(0.2×8)+(0.5×2)+(0.3×−4)$$

$$EV=1.6+1.0+(−1.2)=1.4$$

---

### Exercise 8.2
Suppose Max has two actions:

- `a1` leads directly to utility `3`
- `a2` leads to the chance node above

Which action should Max choose?

**Your answer:**

Max chooses a1 with utility 3, since 3 > 1.4

---

### Exercise 8.3
Why is expected value used at chance nodes instead of max or min?

**Your answer:**

Because chance nodes represent outcomes controlled by nature (randomness) — not by Max or Min. Neither player chooses which outcome occurs, so neither max nor min is appropriate.

Expected value is the mathematically correct way to reason about random events: it weights each possible outcome by how likely it is to actually occur. A rational agent acting under uncertainty should prefer the action that maximizes average outcome over many encounters with the same chance event — which is exactly what expected value captures.

Using max would be overly optimistic (assuming nature always cooperates), and using min would be overly pessimistic (assuming nature always works against you). Expected value is the neutral, probabilistically honest aggregate.

---

### Exercise 8.4
Compute the expected value of the chance node.

In [9]:
probs = [0.2, 0.5, 0.3]
values = [8, 2, -4]

# TODO: compute expected value
expected_value = sum(p * v for p, v in zip(probs, values))
print(expected_value)

# Compare action a1 = 3 with action a2 = expected_value
# Which is better for Max?
a1 = 3
a2 = expected_value
if a1 > a2:
    print("a1 is better for Max")
elif a1 < a2:
    print("a2 is better for Max")
else:    print("a1 and a2 are equally good for Max")


1.4000000000000001
a1 is better for Max


## Part 9. MCTS for Stochastic Games

The lecture notes state that MCTS can also be applied to stochastic games because random actions can be added naturally to playouts.

### Exercise 9.1
What makes MCTS attractive for stochastic games?

**Your answer:**

MCTS handles stochasticity naturally and without modification to its core structure. When a random event occurs during a playout, it is simply sampled from its probability distribution and the simulation continues. 

There is no need to explicitly enumerate all possible chance outcomes or compute their probabilities — the randomness is absorbed into the simulation process itself. This makes MCTS easy to apply to any stochastic game as long as you can simulate it, even if the transition probabilities are unknown or complex.

---

### Exercise 9.2
What new difficulty appears when random actions make the game tree wider?

**Your answer:**

In stochastic games, each chance node branches into all possible random outcomes, multiplying the branching factor significantly. A tree that was already wide from player actions becomes exponentially wider when chance events are layered on top. This means:

- Many more nodes exist at each depth level
- The same total playout budget is spread thinner across a much larger tree
- Individual nodes receive fewer visits, making their value estimates less reliable

---

### Exercise 9.3
Why may MCTS need many more playouts in stochastic games?

**Your answer:**

In a deterministic game, visiting a node repeatedly refines its estimate steadily. In a stochastic game, different playouts through the same node may encounter different random outcomes, introducing variance into the estimate. 

To average out this randomness and get a stable, accurate win rate estimate, many more samples are required. The higher the variance of the chance outcomes, the more playouts are needed to converge — this is essentially the same reason statistical estimates require larger sample sizes when data is noisy.

---

### Exercise 9.4
Compare the following two methods for stochastic games:

1. Heuristic Expectiminimax
2. MCTS

Fill in the table:

Criterion | Heuristic Expectiminimax | MCTS |
----------|---------------------------|------
Needs evaluation function | Yes | No (but can be used to improve playouts)
Uses random playouts | No | Yes
Handles large branching better | No | Yes
Easy to explain mathematically | Yes | No (more complex due to tree search and playouts)


## Part 10. Reflection and Discussion

### Discussion 10.1
A move may look weak in the short term but still be strong in the long term. Why can this happen in adversarial search?

**Your answer:**

This happens because adversarial search involves sequential, interdependent decisions — the value of a move is not determined by its immediate result but by the entire line of play it initiates.

Several concrete reasons:

- Sacrifice patterns. In chess, sacrificing a piece (losing material immediately) can open lines, expose the enemy king, or force a winning tactical sequence several moves later. The shallow evaluation sees only the material loss.

- Positional investment. A move that concedes a small tactical point may establish a structurally dominant position — better piece coordination, space control — whose payoff compounds over many future turns.

- The horizon effect. A heuristic cutoff at depth dd
d simply cannot see consequences at depth d+1d+1
d+1 and beyond. A move that delays a disaster just past the horizon looks falsely safe; a move that sets up a long combination looks falsely weak.

- Opponent-dependent value. The strength of a move depends on how the opponent responds. A move that looks passive may be optimal precisely because it denies the opponent every good continuation — something only visible by searching deeper into the opponent's reply tree.

This is fundamentally why depth matters in adversarial search, and why evaluation functions are imperfect substitutes for full lookahead.

---

### Discussion 10.2
When would you prefer MCTS over heuristic alpha-beta search?

**Your answer:**

When the search space is very large and it's difficult to design a good heuristic evaluation function, MCTS can be more effective as it relies on random sampling and does not require a heuristic.

---

### Discussion 10.3
When would you prefer heuristic alpha-beta search over MCTS?

**Your answer:**

When the game tree is not too large and an evaluation function can be defined effectively.

---

### Discussion 10.4
Which is more dangerous in practice:

- a poor evaluation function, or
- aggressive forward pruning?

Explain your reasoning.

**Your answer:**

A poor evaluation function can lead to consistently bad decisions, as it may not accurately reflect the true value of game states. This can cause the algorithm to choose suboptimal moves, leading to losses. On the other hand, aggressive forward pruning can potentially miss critical moves or strategies, but if the evaluation function is good, it may still guide the algorithm towards strong moves. 

Therefore, a poor evaluation function is generally more dangerous in practice because it can systematically misguide the search process, while aggressive pruning may still allow for some good moves to be considered if the evaluation function is effective.

## Part 11. Mini Challenge

Design your own very small game scenario and answer the following:

1. What are the legal actions?
2. What is one possible evaluation function?
3. What type of uncertainty, if any, exists?
4. Would you solve it using heuristic alpha-beta search, expectiminimax, or MCTS?
5. Why?

**Write your mini design here:**

## Submission Checklist

Before submitting, make sure you have:

- answered all short-response questions,
- completed the numerical exercises,
- filled in the code cells,
- explained your reasoning clearly,
- reflected on the tradeoffs between methods.
